[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C19_Bayesian_ML_Course/04_gaussian_processes/04_gaussian_processes.ipynb)

# 04 · 高斯过程 GP（纯 numpy 从零，Cholesky）

目标：从零实现 **核函数**（RBF/Matern）、**GP 后验**（均值/方差）、**对数边际似然**与**超参优化**，用 **Cholesky** 路径并与「直接套公式（含求逆）」对拍。

路线：RBF/Matern 核矩阵（半正定）→ 从 GP 先验采样 → Cholesky GP 回归（对拍闭式）→ 后验方差不依赖 y、稀疏处膨胀 → 对数边际似然 → 超参网格优化 → GP 后验均值 == 核岭回归 → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：**GP = 函数上的高斯**。后验均值 = 训练目标的核加权平均；后验方差只看『离训练点多远』、不看 y。Cholesky 把 $(K+\sigma_n^2 I)^{-1}$ 的求解做得又稳又省。

## 1 · 核函数与核矩阵（RBF / Matern，半正定）

RBF: $k=\sigma_f^2\exp(-r^2/2\ell^2)$；Matern-5/2 有闭式。核矩阵 $K_{ij}=k(x_i,x_j)$ 必须**半正定**（特征值≥0）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def check_allclose(name, got, ref, atol=1e-8, rtol=1e-5):
    got=np.asarray(got,float); ref=np.asarray(ref,float)
    ok=np.allclose(got,ref,atol=atol,rtol=rtol)
    err=float(np.max(np.abs(got-ref))) if got.size else 0.0
    print(f'[{name:<32}] allclose={ok}  max|err|={err:.2e}')
    assert ok, f'{name} 不一致'; return ok

def sqdist(A, B):
    '''成对平方距离矩阵 |a_i - b_j|^2。A:(n,d) B:(m,d) -> (n,m)。'''
    A = np.atleast_2d(A); B = np.atleast_2d(B)
    return np.sum(A**2,1)[:,None] + np.sum(B**2,1)[None,:] - 2*A@B.T

def rbf_kernel(A, B, ell=1.0, sigma_f=1.0):
    return sigma_f**2 * np.exp(-0.5 * np.maximum(sqdist(A,B), 0.0) / ell**2)

def matern52_kernel(A, B, ell=1.0, sigma_f=1.0):
    r = np.sqrt(np.maximum(sqdist(A,B), 0.0))
    s = np.sqrt(5.0) * r / ell
    return sigma_f**2 * (1 + s + s**2/3.0) * np.exp(-s)

X = np.linspace(-3, 3, 12).reshape(-1, 1)
for name, kfn in [('RBF', rbf_kernel), ('Matern52', matern52_kernel)]:
    Kmat = kfn(X, X, ell=1.0, sigma_f=1.0)
    assert np.allclose(Kmat, Kmat.T), f'{name} 核矩阵应对称'
    eigs = np.linalg.eigvalsh(Kmat)
    assert eigs.min() > -1e-8, f'{name} 核矩阵应半正定'
    assert np.allclose(np.diag(Kmat), 1.0), f'{name} 对角应=sigma_f^2=1'
    print(f'{name}: 对称✅ 半正定✅ (最小特征值={eigs.min():.2e})')
print('✅ 核矩阵对称半正定 —— 合法的 GP 协方差')

## 2 · 从 GP 先验采样：核决定函数的样子

GP 先验在网格点上就是 $\mathcal N(0, K)$。用 Cholesky $K=LL^\top$ 采样 $f=L\,z$（$z\sim\mathcal N(0,I)$）。不同长度尺度 $\ell$ 给出不同波动快慢的函数——直观看到核如何编码先验。

In [ ]:
def sample_gp_prior(X, kfn, n_samples, rng, jitter=1e-8, **kw):
    K = kfn(X, X, **kw) + jitter*np.eye(len(X))
    L = np.linalg.cholesky(K)
    z = rng.standard_normal((len(X), n_samples))
    return L @ z                          # (n, n_samples)，每列一个先验函数样本

Xg = np.linspace(-5, 5, 100).reshape(-1, 1)
# 短 ell -> 抖动快; 长 ell -> 平缓。用样本的相邻差分方差粗测波动
f_short = sample_gp_prior(Xg, rbf_kernel, 50, np.random.default_rng(1), ell=0.3)
f_long  = sample_gp_prior(Xg, rbf_kernel, 50, np.random.default_rng(1), ell=3.0)
rough_short = np.mean(np.var(np.diff(f_short, axis=0), axis=0))
rough_long  = np.mean(np.var(np.diff(f_long, axis=0), axis=0))
print(f'短 ell=0.3 相邻差分方差={rough_short:.4f} (抖动大)')
print(f'长 ell=3.0 相邻差分方差={rough_long:.4f} (平缓)')
assert rough_short > rough_long, '短长度尺度 -> 函数更抖'
# 先验样本均值≈0、方差≈sigma_f^2=1
assert abs(f_short.mean()) < 0.2
assert abs(np.var(f_short) - 1.0) < 0.2
print('✅ 核的长度尺度直接决定先验函数的波动快慢')

## 3 · GP 回归（Cholesky）对拍直接公式

实现两条路：① **Cholesky 路径**（Rasmussen 算法 2.1，数值稳定）；② **直接套公式**（含显式求逆，作参照）。对拍它们的后验均值/方差逐位一致。

In [ ]:
def gp_regression_cholesky(X, y, Xs, kfn, sigma_n=0.1, jitter=1e-8, **kw):
    '''Rasmussen & Williams 算法 2.1。返回 (后验均值, 后验方差)。'''
    n = len(X)
    K = kfn(X, X, **kw) + (sigma_n**2 + jitter)*np.eye(n)
    L = np.linalg.cholesky(K)                       # K = L Lᵀ
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))   # (K)⁻¹ y
    Ks = kfn(X, Xs, **kw)                           # (n, m)
    Kss = kfn(Xs, Xs, **kw)                         # (m, m)
    mean = Ks.T @ alpha
    v = np.linalg.solve(L, Ks)                      # (n, m)
    var = np.diag(Kss) - np.sum(v**2, axis=0)
    return mean, var

def gp_regression_naive(X, y, Xs, kfn, sigma_n=0.1, **kw):
    '''直接套公式（显式求逆，作对拍参照，勿用于生产）。'''
    n = len(X)
    Kinv = np.linalg.inv(kfn(X,X,**kw) + sigma_n**2*np.eye(n))
    Ks = kfn(X, Xs, **kw); Kss = kfn(Xs, Xs, **kw)
    mean = Ks.T @ Kinv @ y
    var = np.diag(Kss) - np.sum((Ks.T @ Kinv) * Ks.T, axis=1)
    return mean, var

# 造数据: y = sin(x) + 噪声
Xtr = np.linspace(-4, 4, 15).reshape(-1, 1)
ytr = np.sin(Xtr).ravel() + 0.1*rng.standard_normal(15)
Xte = np.linspace(-5, 5, 60).reshape(-1, 1)
m_chol, v_chol = gp_regression_cholesky(Xtr, ytr, Xte, rbf_kernel, sigma_n=0.1, ell=1.0)
m_naive, v_naive = gp_regression_naive(Xtr, ytr, Xte, rbf_kernel, sigma_n=0.1, ell=1.0)
check_allclose('GP 均值 Cholesky vs 直接公式', m_chol, m_naive, atol=1e-6)
check_allclose('GP 方差 Cholesky vs 直接公式', v_chol, v_naive, atol=1e-6)
assert np.all(v_chol > -1e-9), '后验方差应非负'
print('✅ Cholesky 路径与直接公式逐位一致（且 Cholesky 更稳更省）')

## 4 · 后验方差不依赖 y、在数据稀疏处膨胀

GP 的招牌性质：后验方差**只看测试点离训练点多远**，与观测值 $y$ 无关。验证：①换一组 $y$ 方差不变；②训练点附近方差小、远离训练点方差回升到先验。

In [ ]:
# 同样的 X, 不同的 y -> 后验方差应完全相同
y1 = np.sin(Xtr).ravel()
y2 = (Xtr.ravel()**2) * 0.1            # 完全不同的目标
_, var1 = gp_regression_cholesky(Xtr, y1, Xte, rbf_kernel, sigma_n=0.1, ell=1.0)
_, var2 = gp_regression_cholesky(Xtr, y2, Xte, rbf_kernel, sigma_n=0.1, ell=1.0)
check_allclose('后验方差不依赖 y', var1, var2, atol=1e-10)

# 数据稀疏处方差更大：在训练点集中区(|x|<4)外 vs 内
Xtr_dense = np.linspace(-2, 2, 12).reshape(-1, 1)   # 只在 [-2,2] 有数据
ytr_dense = np.sin(Xtr_dense).ravel()
Xfar = np.array([[0.0], [5.0]])                     # 一个在数据内、一个在外
_, var_test = gp_regression_cholesky(Xtr_dense, ytr_dense, Xfar, rbf_kernel, sigma_n=0.1, ell=1.0)
print(f'x=0 (数据密集) 后验方差={var_test[0]:.4f}')
print(f'x=5 (远离数据) 后验方差={var_test[1]:.4f}')
assert var_test[1] > var_test[0], '远离数据处方差应更大'
# 远处方差应回到先验方差 sigma_f^2=1
assert abs(var_test[1] - 1.0) < 0.05, '极远处后验方差→先验方差'
print('✅ 后验方差不依赖 y、在数据稀疏处膨胀回先验 —— GP 自动不确定性量化')

## 5 · 对数边际似然（Cholesky 算 log-det）

$$ \log p(y\mid X,\theta)=-\tfrac12 y^\top(K+\sigma_n^2 I)^{-1}y - \tfrac12\log|K+\sigma_n^2 I| - \tfrac n2\log 2\pi $$

用 Cholesky：$y^\top(K+\sigma_n^2I)^{-1}y=\|L^{-1}y\|^2$（即 $\alpha^\top y$），$\log|K+\sigma_n^2I|=2\sum\log L_{ii}$。对拍：用 `np.linalg.slogdet` 独立算 log-det 验证。

In [ ]:
def log_marginal_likelihood(X, y, kfn, sigma_n=0.1, jitter=1e-8, **kw):
    n = len(X)
    Ky = kfn(X, X, **kw) + (sigma_n**2 + jitter)*np.eye(n)
    L = np.linalg.cholesky(Ky)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    data_fit = -0.5 * y @ alpha
    complexity = -np.sum(np.log(np.diag(L)))        # = -0.5 log|Ky|
    const = -0.5 * n * np.log(2*np.pi)
    return data_fit + complexity + const

lml = log_marginal_likelihood(Xtr, ytr, rbf_kernel, sigma_n=0.1, ell=1.0)
print(f'对数边际似然 = {lml:.4f}')

# 对拍：用 slogdet 独立算 log|Ky| 与二次型
Ky = rbf_kernel(Xtr, Xtr, ell=1.0) + 0.1**2*np.eye(15) + 1e-8*np.eye(15)
sign, logdet = np.linalg.slogdet(Ky)
quad = ytr @ np.linalg.solve(Ky, ytr)
lml_ref = -0.5*quad - 0.5*logdet - 0.5*15*np.log(2*np.pi)
check_allclose('LML Cholesky vs slogdet', lml, lml_ref, atol=1e-6)
print('✅ 对数边际似然（Cholesky 路径）= 直接 slogdet，数值一致')

## 6 · 超参优化：最大化边际似然（网格 + 奥卡姆）

边际似然自动权衡拟合与复杂度。网格搜索 $(\ell,\sigma_n)$ 找最大边际似然的超参，并验证它能恢复出合理的长度尺度（既不过拟合也不欠拟合）。

In [ ]:
# 数据来自 ell≈1 的平滑函数 + 适度噪声
Xo = np.linspace(-5, 5, 25).reshape(-1, 1)
yo = np.sin(Xo).ravel() + 0.15*rng.standard_normal(25)

ells = np.linspace(0.2, 4.0, 25)
noises = np.linspace(0.05, 0.6, 20)
best_lml, best = -np.inf, None
lml_grid = np.zeros((len(ells), len(noises)))
for i, el in enumerate(ells):
    for j, sn in enumerate(noises):
        L = log_marginal_likelihood(Xo, yo, rbf_kernel, sigma_n=sn, ell=el)
        lml_grid[i, j] = L
        if L > best_lml:
            best_lml, best = L, (el, sn)
print(f'最优超参: ell={best[0]:.2f}, sigma_n={best[1]:.3f}, LML={best_lml:.3f}')
# 恢复的长度尺度应在合理范围（不接近极小/极大边界 -> 没有过/欠拟合）
assert 0.4 < best[0] < 3.0, '最优长度尺度应落在合理中段'
assert best[1] < 0.4, '应识别出适度噪声（接近真值 0.15）'
# 极小 ell（过拟合）的 LML 应低于最优
lml_overfit = log_marginal_likelihood(Xo, yo, rbf_kernel, sigma_n=0.15, ell=0.2)
assert lml_overfit < best_lml, '过小长度尺度(过拟合) LML 应更低 -> 奥卡姆惩罚'
print('✅ 边际似然自动选出恰当复杂度（奥卡姆剃刀），惩罚过拟合')

---
## ✏️ 练习 1：RBF 核矩阵

实现 `rbf_kernel(A, B, ell, sigma_f)` 返回核矩阵，验证对称、半正定、对角=$\sigma_f^2$、且 $\ell$ 越大远处核值越大（相关性衰减更慢）。

In [ ]:
def sqdist(A, B):
    A = np.atleast_2d(A); B = np.atleast_2d(B)
    return np.sum(A**2,1)[:,None] + np.sum(B**2,1)[None,:] - 2*A@B.T

def rbf_kernel(A, B, ell=1.0, sigma_f=1.0):
    # TODO: sigma_f^2 * exp(-0.5 * sqdist / ell^2)（注意 sqdist 可能有微小负值，clip 到 0）
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
X = np.linspace(0, 5, 10).reshape(-1, 1)
K = rbf_kernel(X, X, ell=1.0, sigma_f=2.0)
assert np.allclose(K, K.T), '对称'
assert np.linalg.eigvalsh(K).min() > -1e-8, '半正定'
assert np.allclose(np.diag(K), 4.0), '对角=sigma_f^2=4'
# 两个相距 2 的点：ell 越大核值越大
k_small_ell = rbf_kernel(np.array([[0.0]]), np.array([[2.0]]), ell=0.5)[0,0]
k_large_ell = rbf_kernel(np.array([[0.0]]), np.array([[2.0]]), ell=3.0)[0,0]
assert k_large_ell > k_small_ell, 'ell 越大 -> 远处相关性越强'
print('✅ 练习 1 通过：RBF 核对称半正定，ell 控制相关衰减')

## ✏️ 练习 2：GP 后验（Cholesky）

实现 `gp_posterior(X, y, Xs, kfn, sigma_n, **kw)` 返回后验 `(mean, var)`，用 Cholesky 路径。验证：噪声→0 时后验均值在训练点处**穿过** $y$、训练点方差→0（插值）。

In [ ]:
def gp_posterior(X, y, Xs, kfn, sigma_n=0.1, jitter=1e-8, **kw):
    # TODO: Cholesky 算 alpha=(K+σ²I)⁻¹y; mean=Ks.T@alpha;
    #       v=solve(L,Ks); var=diag(Kss)-sum(v²)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
Xtr = np.array([[-2.],[0.],[2.]]); ytr = np.array([1.0, -1.0, 0.5])
# 噪声极小 -> 在训练点处『插值』：均值≈y、方差≈0
m, v = gp_posterior(Xtr, ytr, Xtr, rbf_kernel, sigma_n=1e-4, ell=1.0)
assert np.allclose(m, ytr, atol=1e-3), '无噪声 GP 应穿过训练点'
assert np.all(v < 1e-3), '训练点处后验方差应≈0'
# 远离训练点方差回升
_, v_far = gp_posterior(Xtr, ytr, np.array([[10.]]), rbf_kernel, sigma_n=1e-4, ell=1.0)
assert v_far[0] > 0.9, '远处方差→先验'
print('✅ 练习 2 通过：无噪声 GP 插值数据、训练点方差→0、远处→先验')

## ✏️ 练习 3：对数边际似然

实现 `log_marginal_likelihood(X, y, kfn, sigma_n, **kw)`（Cholesky）。验证它在「数据来自的真长度尺度」附近取最大（能用于选超参）。

In [ ]:
def log_marginal_likelihood(X, y, kfn, sigma_n=0.1, jitter=1e-8, **kw):
    # TODO: data_fit=-0.5 y@alpha; complexity=-Σlog diag(L); const=-0.5 n log2π
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
rng_t = np.random.default_rng(5)
# 数据来自 ell=1.5 的 GP 先验
Xd = np.linspace(-5,5,40).reshape(-1,1)
Ktrue = rbf_kernel(Xd, Xd, ell=1.5) + 1e-6*np.eye(40)
yd = np.linalg.cholesky(Ktrue) @ rng_t.standard_normal(40) + 0.1*rng_t.standard_normal(40)
# 在真 ell 附近 LML 应高于明显错误的 ell
lml_true = log_marginal_likelihood(Xd, yd, rbf_kernel, sigma_n=0.1, ell=1.5)
lml_tiny = log_marginal_likelihood(Xd, yd, rbf_kernel, sigma_n=0.1, ell=0.1)
lml_huge = log_marginal_likelihood(Xd, yd, rbf_kernel, sigma_n=0.1, ell=20.0)
assert lml_true > lml_tiny, '真 ell 的 LML 应 > 过小 ell（过拟合）'
assert lml_true > lml_huge, '真 ell 的 LML 应 > 过大 ell（欠拟合）'
print(f'✅ 练习 3 通过：LML 在真 ell=1.5 附近最高 (LML={lml_true:.1f} > {lml_tiny:.1f}, {lml_huge:.1f})')

## ✏️ 练习 4：超参网格搜索

实现 `optimize_length_scale(X, y, kfn, ells, sigma_n)`：在给定 $\ell$ 网格上返回边际似然最大的 $\ell$。这是 type-II MLE / 经验贝叶斯的最简形式。

In [ ]:
def optimize_length_scale(X, y, kfn, ells, sigma_n=0.1):
    # TODO: 对每个 ell 算 LML，返回 argmax 的 ell
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
ells = np.linspace(0.2, 5.0, 30)
best_ell = optimize_length_scale(Xd, yd, rbf_kernel, ells, sigma_n=0.1)
# 数据来自 ell=1.5，恢复的应在其附近
assert 0.8 < best_ell < 3.0, f'恢复的 ell 应接近真值 1.5, 得 {best_ell:.2f}'
# 返回的确实是最大 LML 的 ell
lmls = [log_marginal_likelihood(Xd, yd, rbf_kernel, sigma_n=0.1, ell=e) for e in ells]
assert abs(best_ell - ells[np.argmax(lmls)]) < 1e-12
print(f'✅ 练习 4 通过：网格搜出 ell={best_ell:.2f}（真值 1.5）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def rbf_kernel(A, B, ell=1.0, sigma_f=1.0):
    return sigma_f**2 * np.exp(-0.5 * np.maximum(sqdist(A, B), 0.0) / ell**2)

In [ ]:
# 练习 2 参考答案
def gp_posterior(X, y, Xs, kfn, sigma_n=0.1, jitter=1e-8, **kw):
    n = len(X)
    K = kfn(X, X, **kw) + (sigma_n**2 + jitter)*np.eye(n)
    L = np.linalg.cholesky(K)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    Ks = kfn(X, Xs, **kw); Kss = kfn(Xs, Xs, **kw)
    mean = Ks.T @ alpha
    v = np.linalg.solve(L, Ks)
    var = np.diag(Kss) - np.sum(v**2, axis=0)
    return mean, var

In [ ]:
# 练习 3 参考答案
def log_marginal_likelihood(X, y, kfn, sigma_n=0.1, jitter=1e-8, **kw):
    n = len(X)
    Ky = kfn(X, X, **kw) + (sigma_n**2 + jitter)*np.eye(n)
    L = np.linalg.cholesky(Ky)
    alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
    return -0.5*y@alpha - np.sum(np.log(np.diag(L))) - 0.5*n*np.log(2*np.pi)

In [ ]:
# 练习 4 参考答案
def optimize_length_scale(X, y, kfn, ells, sigma_n=0.1):
    lmls = [log_marginal_likelihood(X, y, kfn, sigma_n=sigma_n, ell=e) for e in ells]
    return ells[int(np.argmax(lmls))]

---
## 🧪 真实数据胶囊：Mauna Loa CO₂ 趋势的 GP 外推

经典真实数据：夏威夷 Mauna Loa 天文台的大气 CO₂ 月度浓度（Keeling 曲线），是 GP 教科书（Rasmussen & Williams 第 5 章）的标志案例。它有**长期上升趋势 + 年度周期 + 噪声**。我们用 GP 拟合并外推，验证：① GP 抓住趋势；② 外推区不确定性增长。

用真实的 CO₂ 测量值（内置若干年真实月度数据，无需联网）。这里用简单的『长 ell RBF（趋势）』核演示 GP 回归与外推的不确定性。

In [ ]:
# Mauna Loa CO₂ 真实月度均值 (ppm)，2010-2015 选取真实数据点（来源: NOAA/Scripps）
# 内置真实数值（无需联网）；时间以『距 2010.0 的年数』表示
t = np.array([0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0])
co2 = np.array([388.6, 391.0, 391.3, 393.2, 393.9, 396.5,
                396.8, 399.0, 399.7, 401.9, 401.0])   # 真实 CO2 ppm（约值）

# 中心化（GP 假设零均值）
t_mean_in = t.mean(); co2_mean = co2.mean()
tc = (t - t_mean_in).reshape(-1, 1)
yc = co2 - co2_mean

# 用长 ell RBF 抓趋势 + 适度噪声；网格选超参
ells = np.linspace(1.0, 10.0, 30)
best_ell = ells[int(np.argmax([log_marginal_likelihood(tc, yc, rbf_kernel, sigma_n=1.0, ell=e) for e in ells]))]
print(f'选出的趋势长度尺度 ell={best_ell:.2f} 年')

# 在历史 + 未来 3 年上预测
t_pred = np.linspace(-3, 8, 80).reshape(-1, 1)   # 中心化坐标, 含外推到 2018
m, v = gp_regression_cholesky(tc, yc, t_pred, rbf_kernel, sigma_n=1.0, ell=best_ell)
m_ppm = m + co2_mean
std_ppm = np.sqrt(np.maximum(v, 0))

# 验证: 趋势上升
assert m_ppm[-1] > m_ppm[0], 'GP 应抓住 CO2 上升趋势'
# 外推区(未来, 远离数据)不确定性 > 内插区
idx_interp = np.argmin(np.abs(t_pred.ravel() - 0.0))    # 数据中段
idx_extrap = -1                                          # 最远未来
print(f'内插处 (2012) 预测±std = {m_ppm[idx_interp]:.1f} ± {std_ppm[idx_interp]:.2f} ppm')
print(f'外推处 (2018) 预测±std = {m_ppm[idx_extrap]:.1f} ± {std_ppm[idx_extrap]:.2f} ppm')
assert std_ppm[idx_extrap] > std_ppm[idx_interp], '外推不确定性应大于内插'
print('✅ 胶囊验证：GP 抓住 CO2 趋势，且外推到未来时诚实地增大不确定性')

**🧪 胶囊练习**：实现 `gp_predict_interval(X, y, Xs, kfn, sigma_n, z=1.96, **kw)`：返回 GP 预测的 `(mean, lower, upper)` 95% 置信带（`mean ± z*std`）。这把 GP 的后验方差变成可汇报的不确定性区间——GP 相对核岭回归的核心增值。

In [ ]:
def gp_predict_interval(X, y, Xs, kfn, sigma_n=0.1, z=1.96, **kw):
    # TODO: 用 gp_regression_cholesky 得 mean,var; std=sqrt(max(var,0));
    #       返回 (mean, mean - z*std, mean + z*std)
    raise NotImplementedError

In [ ]:
# 自测
mean, lo, hi = gp_predict_interval(tc, yc, t_pred, rbf_kernel, sigma_n=1.0, ell=best_ell)
assert np.all(hi > lo), '上界应大于下界'
assert np.all((mean >= lo) & (mean <= hi)), '均值应在区间内'
# 区间宽度在外推处更大
width = hi - lo
assert width[-1] > width[len(width)//2], '外推处置信带更宽'
print('✅ 胶囊练习通过：GP 给出随距离增宽的 95% 置信带（核岭回归给不了）')

In [ ]:
# 📖 胶囊参考答案
def gp_predict_interval(X, y, Xs, kfn, sigma_n=0.1, z=1.96, **kw):
    mean, var = gp_regression_cholesky(X, y, Xs, kfn, sigma_n=sigma_n, **kw)
    std = np.sqrt(np.maximum(var, 0))
    return mean, mean - z*std, mean + z*std

### 小结
- **GP = 函数上的高斯**：任意有限点上函数值服从多元高斯，由均值（常取 0）+ **核** $k(x,x')$ 完全确定。
- **核**编码先验：RBF（无限光滑）、Matern（$\nu$ 控可微次数，更鲁棒）；**长度尺度 $\ell$** 是最关键超参；核须半正定、可组合。
- **GP 后验**（条件高斯）：均值=训练目标的核加权平均（穿过/接近数据）；**方差不依赖 $y$**、只看离训练点多远、稀疏处膨胀回先验。
- **Cholesky** 是数值基石：$K+\sigma_n^2I=LL^\top$，一份分解供均值/方差/log-det 三用；$O(n^3)$ 是瓶颈；jitter 防数值崩。
- **对数边际似然**有闭式（拟合 − 复杂度 − 常数），最大化它自动选超参（**奥卡姆剃刀**，无需交叉验证）。
- **GP 后验均值 = 核岭回归**，但 GP 多给了**不确定性**——这是它在贝叶斯优化/主动学习里的核心价值。

**下一站**：**模块 05 · 概率图模型** —— 用图结构让高维离散联合分布的精确推断可处理。